In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Step 1: Load raw cross-sectional Silver funding data

silver_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "data" / "curated" / "data_lake" / "silver_fact_funding.parquet"
    if candidate.exists():
        silver_path = candidate
        break

if silver_path is None:
    raise FileNotFoundError("Could not locate data/curated/data_lake/silver_fact_funding.parquet")

funding = pd.read_parquet(silver_path)
required_cols = {"date", "asset_id", "funding_rate_raw_pct"}
missing = required_cols - set(funding.columns)
if missing:
    raise KeyError(f"Missing required columns in silver_fact_funding: {sorted(missing)}")

funding = funding[["date", "asset_id", "funding_rate_raw_pct"]].copy()
funding["date"] = pd.to_datetime(funding["date"], utc=True, errors="raise")
funding = funding.sort_values(["date", "asset_id"]).reset_index(drop=True)

print(f"Loaded {len(funding):,} rows from: {silver_path}")
funding.head()

In [ ]:
# Step 2 + Step 3: Temporal harmonizer + robust environment sensor (rank-based truncated mean)

import numpy as np

# Temporal harmonizer (as specified):
# - date <  2026-01-19 -> already decimal
# - date >= 2026-01-19 -> native percentage, convert to decimal via /100
cutover = pd.Timestamp("2026-01-19T00:00:00+00:00")

funding["harmonized_rate"] = funding["funding_rate_raw_pct"].astype(float)
mask_pct_regime = funding["date"] >= cutover
funding.loc[mask_pct_regime, "harmonized_rate"] = (
    funding.loc[mask_pct_regime, "harmonized_rate"] / 100.0
)

# Original cross-sectional mean (outlier-heavy baseline), annualized
orig_daily = (
    funding.groupby("date", as_index=False)["harmonized_rate"]
    .mean()
    .rename(columns={"harmonized_rate": "funding_mean_dec"})
)
orig_daily["F_tk_apr_original"] = orig_daily["funding_mean_dec"] * 1095.0


def calculate_truncated_mean(group: pd.DataFrame) -> float:
    rates = group["harmonized_rate"].dropna().sort_values().values
    n = len(rates)

    if n == 0:
        return np.nan

    k = int(n / 4)

    if k == 0:
        return float(np.mean(rates))

    middle_coins = rates[k:-k]
    if len(middle_coins) == 0:
        return np.nan

    return float(np.mean(middle_coins))


robust_daily = (
    funding.groupby("date", as_index=False)
    .apply(calculate_truncated_mean, include_groups=False)
    .rename(columns={None: "robust_rate_dec", 0: "robust_rate_dec"})
)

robust_daily["F_tk_apr_environment"] = robust_daily["robust_rate_dec"] * 1095.0
# Forward-fill to handle complete API-offline epochs in final environment series
robust_daily["F_tk_apr_environment"] = robust_daily["F_tk_apr_environment"].ffill()

sensor_compare = orig_daily.merge(
    robust_daily[["date", "F_tk_apr_environment"]], on="date", how="inner"
).sort_values("date")

sensor_compare.tail()

In [ ]:
# Step 4: Visual audit (original mean vs IQM environment sensor)

chart_df = sensor_compare.copy()
chart_df = chart_df[chart_df["date"] <= pd.Timestamp.now(tz="UTC")]

plt.figure(figsize=(14, 6))
plt.plot(
    chart_df["date"],
    chart_df["F_tk_apr_original"] * 100.0,
    label="Original cross-sectional mean (APR)",
    linewidth=1.2,
    alpha=0.8,
)
plt.plot(
    chart_df["date"],
    chart_df["F_tk_apr_environment"] * 100.0,
    label="IQM environment sensor (APR)",
    linewidth=2.0,
)

plt.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
plt.title("Funding Macro Sensor: Original Mean vs Interquartile Mean Environment")
plt.xlabel("Date (UTC)")
plt.ylabel("Annualized Funding (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PM export artifacts: comparison table + chart image
# Overwrites robust_sensor_environment_vs_original.csv with latest calculation

out_dir = Path.cwd() / "out"
out_dir.mkdir(parents=True, exist_ok=True)

export_df = sensor_compare[["date", "F_tk_apr_original", "F_tk_apr_environment"]].copy()
export_df["F_tk_apr_original_pct"] = (export_df["F_tk_apr_original"] * 100.0).round(4)
export_df["F_tk_apr_environment_pct"] = (export_df["F_tk_apr_environment"] * 100.0).round(4)

csv_path = out_dir / "robust_sensor_environment_vs_original.csv"
export_df.to_csv(csv_path, index=False)

png_path = out_dir / "robust_sensor_environment_vs_original.png"
fig = plt.figure(figsize=(14, 6))
plt.plot(
    export_df["date"],
    export_df["F_tk_apr_original"] * 100.0,
    label="Original cross-sectional mean (APR)",
    linewidth=1.2,
    alpha=0.8,
)
plt.plot(
    export_df["date"],
    export_df["F_tk_apr_environment"] * 100.0,
    label="Rank-based truncated environment sensor (APR)",
    linewidth=2.0,
)
plt.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
plt.title("Funding Macro Sensor: Original Mean vs Rank-Based Truncated Environment")
plt.xlabel("Date (UTC)")
plt.ylabel("Annualized Funding (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Saved CSV: {csv_path}")
print(f"Saved PNG: {png_path}")
print("\nHead:")
print(export_df.head().to_string(index=False))

sample_2023 = export_df[export_df["date"].dt.year == 2023].head(10)
print("\n2023 sample:")
print(sample_2023.to_string(index=False))

# Quick sanity: show NaN count in environment series
nan_count = int(export_df["F_tk_apr_environment"].isna().sum())
print(f"\nNaN count in F_tk_apr_environment: {nan_count}")

export_df.tail()

In [ ]:
# Univariate Mandate (SOP Rule 1): histogram + KDE for F_tk_apr_environment

import numpy as np

series_pct = (sensor_compare["F_tk_apr_environment"].dropna() * 100.0)

fig, ax = plt.subplots(figsize=(12, 7), dpi=180)

# Histogram (30 bins)
ax.hist(
    series_pct,
    bins=30,
    density=True,
    alpha=0.55,
    color="steelblue",
    edgecolor="white",
    linewidth=0.6,
    label="Histogram (30 bins)",
)

# KDE overlay (prefer scipy; fallback to pandas density)
try:
    from scipy.stats import gaussian_kde

    kde = gaussian_kde(series_pct)
    x_grid = np.linspace(float(series_pct.min()), float(series_pct.max()), 400)
    ax.plot(x_grid, kde(x_grid), color="darkred", linewidth=2.0, label="KDE")
except Exception:
    series_pct.plot(kind="kde", ax=ax, color="darkred", linewidth=2.0, label="KDE")

# 0% demarcation line
ax.axvline(0.0, color="black", linestyle="--", linewidth=1.2, label="0% boundary")

ax.set_title("Univariate Audit: Robust Macro Sensor (Rank-Based Truncated Mean)")
ax.set_xlabel("F_tk_apr_environment (%)")
ax.set_ylabel("Density")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()

# Save to /notebooks as requested
notebooks_dir = None
for root in [Path.cwd(), *Path.cwd().parents]:
    if root.name == "notebooks":
        notebooks_dir = root
        break
    candidate = root / "notebooks"
    if candidate.exists() and candidate.is_dir():
        notebooks_dir = candidate
        break

if notebooks_dir is None:
    raise FileNotFoundError("Could not locate /notebooks directory for plot export")

hist_path = notebooks_dir / "f_tk_apr_environment_histogram.png"
fig.savefig(hist_path, dpi=180, bbox_inches="tight")
print(f"Saved histogram: {hist_path}")

plt.show()